In [ ]:
import sys
import json
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve repo root whether launched from repo root or Notebooks/
repo_root = Path.cwd().resolve()
if not (repo_root / "backtesting" / "renquant_102").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import common

# ── Strategy configuration ───────────────────────────────────────────────
STRATEGY_DIR = repo_root / "backtesting" / "renquant_102"
MODELS_DIR = STRATEGY_DIR / "models"
config = common.load_strategy_config(STRATEGY_DIR / "strategy_config.json")

WATCHLIST = config["watchlist"]
BENCHMARK = config.get("benchmark", "SPY")
MODEL_NAME = config["model_name"]
PROVIDER = config["data_src"]

# Dynamic 2-year window ending today
END = str(date.today())
START = str(date.today() - timedelta(days=365 * int(config.get("training_years", 2))))

# Full indicator spec — all available indicators computed upfront.
INDICATOR_SPEC = {
    "rsi":         {"period": 14},
    "macd":        {"fast": 12, "slow": 26, "signal": 9},
    "cci":         {"period": 20},
    "bbp":         {"period": 20},
    "stochastic":  {"window": 14, "smooth": 3},
    "adx":         {"period": 14},
    "atr":         {"period": 14},
    "obv":         {"signal_period": 20},
    "williams_r":  {"period": 14},
    "ema":         {"period": 50},
    "momentum":    {"period": 10},
}

# Shared feature columns for Classification and Mean Reversion models.
FEATURE_COLUMNS = ["rsi", "macd_hist", "cci", "bbp", "adx", "williams_r", "obv_slope"]
RATIO_FEATURES = {"rsi", "adx"}
DIFF_FEATURES  = {"macd_hist", "cci", "bbp", "williams_r", "obv_slope"}

# Q-Learning uses trend features (3 features x 5 bins = 375 states)
QL_FEATURE_COLUMNS = ["trend", "rel_mom_20d", "macd_hist"]

# ── Simulation parameters (from config) ──────────────────────────────────
INITIAL_CASH = 10_000
WASH_SALE_DAYS = config.get("wash_sale_days", 30)
MIN_HOLD_DAYS = config.get("min_hold_days", 20)
MAX_HOLD_DAYS = config.get("max_hold_days", 150)
pos_sizing = config.get("position_sizing", {})
MAX_POSITION_PCT = pos_sizing.get("max_position_pct", 0.33)
CASH_RESERVE_PCT = pos_sizing.get("cash_reserve_pct", 0.10)

# ── Approach configurations ──────────────────────────────────────────────
DUAL_MOM_RULES = [
    {"col": "trend",       "buy_above": 1.0,    "sell_below": 0.97},
    {"col": "trend_long",  "buy_above": 1.0,    "sell_below": 0.97},
    {"col": "rel_mom_20d", "buy_above": 0.0,    "sell_below": -0.03},
    {"col": "rel_mom_60d", "buy_above": 0.0,    "sell_below": -0.05},
    {"col": "macd_hist",   "buy_above": 0,       "sell_below": 0},
    {"col": "obv_slope",   "buy_above": 0,       "sell_below": 0},
]

MR_RULES = [
    {"col": "rsi",        "buy_below": 0.9,    "sell_above": 1.1},
    {"col": "bbp",        "buy_below": -0.2,   "sell_above": 0.3},
    {"col": "williams_r", "buy_below": -20,    "sell_above": 20},
    {"col": "trend",      "buy_below": 0.98,   "sell_above": 1.05},
    {"col": "cci",        "buy_below": -30,    "sell_above": 50},
]

CLASSIFICATION_PARAMS = {
    "feature_columns": FEATURE_COLUMNS,
    "lookahead": 10,
    "threshold": 0.04,
    "leaf_size": 25,
    "bags": 15,
    "buy_threshold": 0.1,
    "sell_threshold": -0.1,
}

QLEARNING_PARAMS = {
    "feature_columns": QL_FEATURE_COLUMNS,
    "n_bins": 5,
    "n_epochs": 500,
    "alpha": 0.15,
    "gamma": 0.95,
    "rar": 0.99,
    "radr": 0.9995,
    "dyna": 0,
}

print(f"Strategy:  {MODEL_NAME}")
print(f"Watchlist: {len(WATCHLIST)} symbols")
print(f"Window:    {START} -> {END}")
print(f"Benchmark: {BENCHMARK}")
print(f"Approaches: Dual Momentum, Classification, Q-Learning, Mean Reversion")

In [ ]:
# ── Fetch OHLCV + compute indicators for all watchlist stocks + SPY ──────
all_symbols = list(set(WATCHLIST + [BENCHMARK]))
dfs_raw = {}
dfs_ind = {}

for sym in all_symbols:
    print(f"  Fetching {sym}...", end=" ")
    try:
        raw = common.fetch_ohlcv(sym, start=START, end=END, provider=PROVIDER)
        ind = common.compute_indicators(raw, INDICATOR_SPEC)
        dfs_raw[sym] = raw
        dfs_ind[sym] = ind
        print(f"{len(raw)} bars")
    except Exception as e:
        print(f"FAILED: {e}")

df_spy = dfs_ind[BENCHMARK]
print(f"\nFetched {len(dfs_ind)} / {len(all_symbols)} symbols")

In [ ]:
# ── Build per-symbol relative feature frames ────────────────────────────
# For each watchlist stock, compute:
# - Relative indicator features vs SPY (for Classification, Mean Reversion)
# - Trend features (for Dual Momentum, Q-Learning)
# - Q-Learning variant with rel_price as "close" for relative reward

stock_features = {}    # {symbol: DataFrame with all features}
stock_features_ql = {} # {symbol: DataFrame for Q-Learning (rel_price as close)}
skipped = []

for symbol in WATCHLIST:
    if symbol not in dfs_ind:
        skipped.append(symbol)
        continue

    df_stock = dfs_ind[symbol]

    # Align trading days
    common_idx = df_stock.index.intersection(df_spy.index)
    if len(common_idx) < 100:
        skipped.append(symbol)
        continue

    s = df_stock.loc[common_idx]
    b = df_spy.loc[common_idx]

    df = pd.DataFrame(index=common_idx)
    df["close"] = s["close"]
    df["close_spy"] = b["close"]
    df["rel_price"] = s["close"] / b["close"]

    # Relative indicator features
    for col in FEATURE_COLUMNS:
        if col in RATIO_FEATURES:
            df[col] = s[col] / b[col].replace(0, np.nan)
        else:
            df[col] = s[col] - b[col]

    # Trend-following features
    df["trend"] = s["close"] / s["close"].ewm(span=50, adjust=False).mean()
    df["trend_long"] = s["close"] / s["close"].ewm(span=200, adjust=False).mean()
    df["rel_mom_20d"] = df["rel_price"].pct_change(20)
    df["rel_mom_60d"] = df["rel_price"].pct_change(60)

    df = df.dropna()
    if len(df) < 100:
        skipped.append(symbol)
        continue

    stock_features[symbol] = df

    # Q-Learning variant: rel_price as "close" for relative reward
    df_ql = df[["rel_price", *QL_FEATURE_COLUMNS]].copy()
    df_ql = df_ql.rename(columns={"rel_price": "close"})
    stock_features_ql[symbol] = df_ql

print(f"Built features for {len(stock_features)} symbols")
if skipped:
    print(f"Skipped (insufficient data): {skipped}")

In [ ]:
# ── Helper functions: constraints + equity simulation ────────────────────
# Same logic as renquant_101 — single-stock simulation for model comparison.

def apply_constraints(signals, min_hold_days=0, wash_sale_days=0, max_hold_days=0):
    """Filter raw signals to enforce holding period, max hold, and wash-sale rules."""
    out = signals.values.copy()
    dates = signals.index
    position = 0
    entry_idx = None
    last_sell_idx = None

    for i in range(len(out)):
        sig = out[i]

        if position == 1 and max_hold_days > 0 and entry_idx is not None:
            if (dates[i] - dates[entry_idx]).days >= max_hold_days:
                out[i] = "sell"
                position = 0
                last_sell_idx = i
                entry_idx = None
                continue

        if sig == "buy" and position == 0:
            if last_sell_idx is not None and (dates[i] - dates[last_sell_idx]).days < wash_sale_days:
                out[i] = "hold"
                continue
            position = 1
            entry_idx = i
        elif sig == "sell" and position == 1:
            if entry_idx is not None and (dates[i] - dates[entry_idx]).days < min_hold_days:
                out[i] = "hold"
                continue
            position = 0
            last_sell_idx = i
            entry_idx = None
        else:
            if (sig == "sell" and position == 0) or (sig == "buy" and position == 1):
                out[i] = "hold"
    return pd.Series(out, index=signals.index)


def simulate_equity(signals, prices, initial_cash=INITIAL_CASH,
                    max_position_pct=MAX_POSITION_PCT,
                    cash_reserve_pct=CASH_RESERVE_PCT):
    """Simulate a long-only equity curve with position sizing."""
    close = prices.values
    sigs = signals.values
    cash = float(initial_cash)
    shares = 0
    equity = np.empty(len(sigs))
    trades = []
    entry_idx = None

    for i in range(len(sigs)):
        p = close[i]
        portfolio_value = cash + shares * p

        if sigs[i] == "buy" and shares == 0:
            max_invest = portfolio_value * max_position_pct
            available = cash - portfolio_value * cash_reserve_pct
            invest_amount = min(max_invest, max(available, 0))
            shares = int(invest_amount // p)
            cash -= shares * p
            entry_idx = i
        elif sigs[i] == "sell" and shares > 0:
            cash += shares * p
            shares = 0
            if entry_idx is not None:
                trades.append((signals.index[entry_idx], signals.index[i]))
                entry_idx = None

        equity[i] = cash + shares * p
    return pd.Series(equity, index=signals.index), trades


def compute_sharpe(equity):
    """Annualized Sharpe ratio from an equity series."""
    daily_rets = equity.pct_change().dropna()
    if daily_rets.std() == 0:
        return 0.0
    return daily_rets.mean() / daily_rets.std() * np.sqrt(252)


print("Helper functions defined: apply_constraints, simulate_equity, compute_sharpe")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Per-symbol training loop: 4 approaches, pick best by Sharpe
# ══════════════════════════════════════════════════════════════════════════
# For each symbol, train:
#   1. Dual Momentum (ManualModel — trend/momentum rules)
#   2. Classification (BagLearner/RTLearner random forest)
#   3. Q-Learning (tabular RL with discretized states)
#   4. Mean Reversion (ManualModel — contrarian rules)
# Then simulate equity with constraints, pick best Sharpe.

results = {}  # {symbol: {approach_name: {model, sharpe, return, trades, ...}}}

for i, symbol in enumerate(sorted(stock_features.keys())):
    df = stock_features[symbol]
    df_ql = stock_features_ql[symbol]
    approaches = {}

    # 1. Dual Momentum
    dm_model = common.create_model("manual", score_rules=DUAL_MOM_RULES,
                                   buy_threshold=4, sell_threshold=-3)
    dm_model.train(df)
    dm_sigs = apply_constraints(dm_model.predict_bulk(df),
                                min_hold_days=MIN_HOLD_DAYS,
                                wash_sale_days=WASH_SALE_DAYS,
                                max_hold_days=MAX_HOLD_DAYS)
    dm_eq, dm_trades = simulate_equity(dm_sigs, df["close"])
    approaches["Dual Momentum"] = {
        "model": dm_model, "sharpe": compute_sharpe(dm_eq),
        "equity": dm_eq, "trades": dm_trades,
        "return": dm_eq.iloc[-1] / dm_eq.iloc[0] - 1,
    }

    # 2. Classification (RF)
    clf_model = common.create_model("classification", **CLASSIFICATION_PARAMS)
    clf_model.train(df)
    clf_sigs = apply_constraints(clf_model.predict_bulk(df),
                                 min_hold_days=MIN_HOLD_DAYS,
                                 wash_sale_days=WASH_SALE_DAYS,
                                 max_hold_days=MAX_HOLD_DAYS)
    clf_eq, clf_trades = simulate_equity(clf_sigs, df["close"])
    approaches["Classification"] = {
        "model": clf_model, "sharpe": compute_sharpe(clf_eq),
        "equity": clf_eq, "trades": clf_trades,
        "return": clf_eq.iloc[-1] / clf_eq.iloc[0] - 1,
    }

    # 3. Q-Learning
    ql_model = common.create_model("qlearning", **QLEARNING_PARAMS)
    ql_model.train(df_ql)
    ql_sigs = apply_constraints(ql_model.predict_bulk(df_ql),
                                min_hold_days=MIN_HOLD_DAYS,
                                wash_sale_days=WASH_SALE_DAYS,
                                max_hold_days=MAX_HOLD_DAYS)
    ql_eq, ql_trades = simulate_equity(ql_sigs, df["close"])
    approaches["Q-Learning"] = {
        "model": ql_model, "sharpe": compute_sharpe(ql_eq),
        "equity": ql_eq, "trades": ql_trades,
        "return": ql_eq.iloc[-1] / ql_eq.iloc[0] - 1,
    }

    # 4. Mean Reversion
    mr_model = common.create_model("manual", score_rules=MR_RULES,
                                   buy_threshold=3, sell_threshold=-2)
    mr_model.train(df)
    mr_sigs = apply_constraints(mr_model.predict_bulk(df),
                                min_hold_days=MIN_HOLD_DAYS,
                                wash_sale_days=WASH_SALE_DAYS,
                                max_hold_days=MAX_HOLD_DAYS)
    mr_eq, mr_trades = simulate_equity(mr_sigs, df["close"])
    approaches["Mean Reversion"] = {
        "model": mr_model, "sharpe": compute_sharpe(mr_eq),
        "equity": mr_eq, "trades": mr_trades,
        "return": mr_eq.iloc[-1] / mr_eq.iloc[0] - 1,
    }

    best_name = max(approaches, key=lambda k: approaches[k]["sharpe"])
    results[symbol] = approaches

    print(f"[{i+1:2d}/{len(stock_features)}] {symbol:5s}  "
          f"best={best_name:16s}  Sharpe={approaches[best_name]['sharpe']:.2f}  "
          f"Return={approaches[best_name]['return']:.1%}  "
          f"Trades={len(approaches[best_name]['trades'])}")

print(f"\nTrained {len(results)} symbols, 4 approaches each")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Export best model per symbol
# ══════════════════════════════════════════════════════════════════════════
# For each symbol, export the approach with the highest Sharpe ratio.
# Artifacts go to backtesting/renquant_102/models/{SYMBOL}/

MODELS_DIR.mkdir(parents=True, exist_ok=True)
today_str = str(date.today())

for symbol in sorted(results.keys()):
    approaches = results[symbol]
    best_name = max(approaches, key=lambda k: approaches[k]["sharpe"])
    best = approaches[best_name]

    # Create per-symbol model directory
    model_dir = MODELS_DIR / symbol
    model_dir.mkdir(parents=True, exist_ok=True)

    # Clean old artifacts
    for old in model_dir.glob("*.json"):
        old.unlink()

    # Export the winner (model_name = symbol)
    metadata = best["model"].save(model_dir, symbol)

    # Add trained_date and best_approach to policy-metadata
    meta_path = model_dir / f"{symbol}-policy-metadata.json"
    meta = json.loads(meta_path.read_text())
    meta["trained_date"] = today_str
    meta["best_approach"] = best_name
    meta["sharpe"] = round(best["sharpe"], 4)
    meta_path.write_text(json.dumps(meta, indent=2))

    print(f"  {symbol:5s} -> {best_name:16s}  (Sharpe {best['sharpe']:.2f})  "
          f"policy_type={meta['policy_type']}")

print(f"\nExported {len(results)} models to {MODELS_DIR}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Per-symbol comparison charts (compact grid)
# ══════════════════════════════════════════════════════════════════════════
symbols_sorted = sorted(results.keys())
n_symbols = len(symbols_sorted)
n_cols = 3
n_rows = (n_symbols + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 3.5 * n_rows), sharex=False)
axes_flat = axes.flatten() if n_symbols > 1 else [axes]

approach_colors = {
    "Dual Momentum": "tab:blue",
    "Classification": "tab:orange",
    "Q-Learning": "tab:green",
    "Mean Reversion": "tab:red",
}

for idx, symbol in enumerate(symbols_sorted):
    ax = axes_flat[idx]
    approaches = results[symbol]
    best_name = max(approaches, key=lambda k: approaches[k]["sharpe"])

    df = stock_features[symbol]
    bh = df["close"] / df["close"].iloc[0]
    ax.plot(bh.index, bh, color="gray", linestyle="--", linewidth=0.8, label="Buy&Hold")

    for name, ap in approaches.items():
        norm_eq = ap["equity"] / ap["equity"].iloc[0]
        lw = 2.0 if name == best_name else 0.8
        alpha = 1.0 if name == best_name else 0.4
        ax.plot(norm_eq.index, norm_eq, color=approach_colors[name],
                linewidth=lw, alpha=alpha, label=f"{name}")

    ax.set_title(f"{symbol} — {best_name} (S={approaches[best_name]['sharpe']:.2f})",
                 fontsize=9, fontweight="bold")
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.2)
    if idx == 0:
        ax.legend(fontsize=6, loc="upper left")

# Hide unused axes
for idx in range(n_symbols, len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.suptitle(f"renquant-102: Best approach per symbol ({len(results)} stocks)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cross-symbol summary table
# ══════════════════════════════════════════════════════════════════════════
summary_rows = []
approach_wins = {}

for symbol in sorted(results.keys()):
    approaches = results[symbol]
    best_name = max(approaches, key=lambda k: approaches[k]["sharpe"])
    best = approaches[best_name]
    approach_wins[best_name] = approach_wins.get(best_name, 0) + 1

    hold_days = [(s - b).days for b, s in best["trades"]]
    summary_rows.append({
        "Symbol": symbol,
        "Best Approach": best_name,
        "Sharpe": round(best["sharpe"], 2),
        "Return": f"{best['return']:.1%}",
        "Round-trips": len(best["trades"]),
        "Avg Hold": f"{np.mean(hold_days):.0f}d" if hold_days else "-",
    })

# Add benchmark row
for symbol in sorted(results.keys()):
    df = stock_features[symbol]
    bh_ret = df["close"].iloc[-1] / df["close"].iloc[0] - 1

summary_df = pd.DataFrame(summary_rows).sort_values("Sharpe", ascending=False)

print("Approach win counts:")
for name, count in sorted(approach_wins.items(), key=lambda x: -x[1]):
    print(f"  {name:16s}: {count}")

print(f"\nTop 10 by Sharpe:")
summary_df.head(10)

In [ ]:
# Full summary table
print(f"All {len(summary_df)} symbols:")
summary_df